# 03 — Hyperparameter selection and model training
Train RF and LightGBM models for COD_CANON_V2 and MP_ALL in all three chemical domains. **Inputs:** six archived or newly generated Magpie split tables. **Outputs:** best-parameter registries, trial/checkpoint files, five-fold CV predictions and metrics, final model bundles, holdout predictions and run metadata.

Run cells in order. `RUN` selects the experiment registry; `FORCE=False` reuses completed compatible jobs and resumes supported interrupted jobs. `FORCE=True` replaces the corresponding jobs. Preserve archived PAPER_V2 artifacts before starting another experiment.

Hyperopt TPE uses 50 evaluations, seed 42 and inner fold 0 for selection by ROC-AUC; test records are excluded. Subsequent five-fold CV is **not fully nested**, because tuning has already used one inner validation fold. Tuning classifiers are unweighted; the saved CV/final builders use `class_weight="balanced"`. This distinction is part of the PAPER_V2 protocol. See [Methods](../docs/METHODS.md).


In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ["TQDM_NOTEBOOK"] = "0"

from project_config import make_cfg
from private_paths import COD_DIR, MP_DIR
from pipelines.hyper_tuning_pipeline import HyperTuningPipeline, HyperTuningPipelineConfig
from pipelines.train_pipeline import TrainPipeline

RUN = "PAPER_V2"
FORCE = False
MODELS = ("rf", "lgbm")
STEPS = ("magpie",)
DOMAINS = ("N", "O", "M")
TARGET_SPEC = {
    "source_col": "centrosymmetry",
    "out_col": "y",
    "mapping": {"C": 0, "N": 1},
    "allow_values": ["C", "N"],
}

tune_cfg = HyperTuningPipelineConfig(
    fold_col="fold",
    is_test_col="is_test",
    max_evals=50,
    selected_fold=0,
    metric="auc",
    random_state=42,
    show_progress=True,
    save_trials=True,
    save_registry=True,
)

def project_cfg(project_name):
    root = COD_DIR if project_name == "COD_CANON_V2" else MP_DIR
    return make_cfg(project_name=project_name, root_cif=root, log_time=True)

def registry_path(cfg):
    return Path(cfg.PROJECT_DIR) / "tuning" / f"best_params_{RUN}.json"

def show_best_params(project_name):
    cfg = project_cfg(project_name)
    path = registry_path(cfg)
    registry = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for entry in registry.get("entries", []):
        row = dict(entry.get("key") or {})
        row.update(entry.get("params") or {})
        tuning = entry.get("tuning") or {}
        row["tune_run_id"] = tuning.get("tune_run_id")
        row["max_evals"] = tuning.get("max_evals")
        rows.append(row)
    table = pd.DataFrame(rows)
    display(table)
    print(path)
    return table


## COD hyperparameter selection
Tune both model families in domains N/O/M and display the saved parameters.


In [ ]:
cfg = project_cfg("COD_CANON_V2")
res_tune = HyperTuningPipeline(cfg, tune_cfg).run(
    models=MODELS,
    steps=STEPS,
    domains=DOMAINS,
    mode="cv",
    run=RUN,
    target_spec=TARGET_SPEC,
    force=FORCE,
)
print("Registry:", res_tune.get("registry_path"))
print("Jobs:", len(res_tune["jobs"]))


In [ ]:
show_best_params("COD_CANON_V2")


## MP hyperparameter selection
Apply the same selection protocol to the three MP domains.


In [ ]:
cfg = project_cfg("MP_ALL")
res_tune = HyperTuningPipeline(cfg, tune_cfg).run(
    models=MODELS,
    steps=STEPS,
    domains=DOMAINS,
    mode="cv",
    run=RUN,
    target_spec=TARGET_SPEC,
    force=FORCE,
)
print("Registry:", res_tune.get("registry_path"))
print("Jobs:", len(res_tune["jobs"]))


In [ ]:
show_best_params("MP_ALL")


## COD cross-validation
Use the selected parameters in five non-test validation folds. Exclude the pooled `ALL` row when calculating fold mean and sample standard deviation.


In [ ]:
cfg = project_cfg("COD_CANON_V2")
TrainPipeline(cfg).run(
    models=MODELS,
    steps=STEPS,
    domains=DOMAINS,
    mode="cv",
    run=RUN,
    target_name="centrosymmetry_CN",
    params_registry_path=str(registry_path(cfg)),
    use_registry=True,
    force=FORCE,
)


## COD final models
Fit on all non-test records and evaluate once on the fixed test set. Positive probabilities refer to noncentrosymmetry; classification metrics use threshold 0.5.


In [ ]:
cfg = project_cfg("COD_CANON_V2")
TrainPipeline(cfg).run(
    models=MODELS,
    steps=STEPS,
    domains=DOMAINS,
    mode="final",
    run=RUN,
    target_name="centrosymmetry_CN",
    params_registry_path=str(registry_path(cfg)),
    use_registry=True,
    force=FORCE,
)


## MP cross-validation
Use the selected parameters in five non-test validation folds. Exclude the pooled `ALL` row when calculating fold mean and sample standard deviation.


In [ ]:
cfg = project_cfg("MP_ALL")
TrainPipeline(cfg).run(
    models=MODELS,
    steps=STEPS,
    domains=DOMAINS,
    mode="cv",
    run=RUN,
    target_name="centrosymmetry_CN",
    params_registry_path=str(registry_path(cfg)),
    use_registry=True,
    force=FORCE,
)


## MP final models
Fit on all non-test records and evaluate once on the fixed test set. Positive probabilities refer to noncentrosymmetry; classification metrics use threshold 0.5.


In [ ]:
cfg = project_cfg("MP_ALL")
TrainPipeline(cfg).run(
    models=MODELS,
    steps=STEPS,
    domains=DOMAINS,
    mode="final",
    run=RUN,
    target_name="centrosymmetry_CN",
    params_registry_path=str(registry_path(cfg)),
    use_registry=True,
    force=FORCE,
)


## Run selection for reports
Each project/model/domain job has its own timestamped run ID and `meta_<run_id>.json`. Reporting notebooks select matching metadata and stop on ambiguity. Set their `RUN_OVERRIDES` to the desired exact run IDs when multiple runs exist. Retain metadata together with predictions, models and the source split files.
